# 🧠 Aula 08: Autoencoders Convolucionais, Espaço Latente e Detecção de Anomalias

**Curso:** Redes Neurais Profundas — Graduação Faculdade Infnet  
**Conexão com a Avaliação:** Gabarito prático e conceitual completo do **Estudo de Caso 2 (Autoencoder & Anomalias)**.  

---

### 🎯 Objetivos de Aprendizagem
1. Construir e treinar um **Autoencoder Denso** e um **Autoencoder Convolucional (ConvAutoencoder)** no PyTorch.
2. Comparar a fidelidade de reconstrução e calcular a razão de compressão dimensional do gargalo ($\text{Dim}(Z) / \text{Dim}(X)$).
3. Extrair as representações compactas do **Espaço Latente** e visualizar os agrupamentos com **t-SNE** e **PCA**.
4. Implementar um sistema de **Detecção de Anomalias por Erro de Reconstrução MSE**.
5. Avaliar criticamente a seleção de limiares (*thresholds*): a falha da regra ingênua $\mu + 2\sigma$ vs. calibração orientada por curvas ROC-AUC / Precision-Recall com conjunto rotulado de validação.


## 1. Configuração do Ambiente e Importação de Bibliotecas

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.metrics import roc_curve, auc, precision_recall_curve, classification_report, confusion_matrix

# Configurar dispositivo (GPU se disponível)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Utilizando dispositivo: {device}")

## 2. Preparação do Dataset (MNIST)

In [ ]:
# Transformação para converter imagem para Tensor e normalizar entre [0, 1]
transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

print(f"Total de imagens de treino: {len(train_dataset)}")
print(f"Total de imagens de teste: {len(test_dataset)}")

## 3. Implementação das Arquiteturas: Denso vs. Convolucional

### 3.1 Autoencoder Denso (MLP)

In [ ]:
class DenseAutoencoder(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        # Input: 1 x 28 x 28 = 784
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 784),
            nn.Sigmoid(),
            nn.Unflatten(1, (1, 28, 28))
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat

dense_model = DenseAutoencoder(latent_dim=32).to(device)
print(f"Parâmetros do Autoencoder Denso: {sum(p.numel() for p in dense_model.parameters()):,}")

### 3.2 Autoencoder Convolucional (ConvAutoencoder)

In [ ]:
class ConvAutoencoder(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        # Encoder
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1),  # -> 16 x 14 x 14
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1), # -> 32 x 7 x 7
            nn.BatchNorm2d(32),
            nn.ReLU()
        )
        self.fc_encoder = nn.Linear(32 * 7 * 7, latent_dim)
        
        # Decoder
        self.fc_decoder = nn.Linear(latent_dim, 32 * 7 * 7)
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1), # -> 16 x 14 x 14
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.ConvTranspose2d(16, 1, kernel_size=3, stride=2, padding=1, output_padding=1),  # -> 1 x 28 x 28
            nn.Sigmoid()
        )

    def encode(self, x):
        h = self.encoder_conv(x)
        h = h.view(h.size(0), -1)
        z = self.fc_encoder(h)
        return z

    def decode(self, z):
        h = self.fc_decoder(z)
        h = h.view(h.size(0), 32, 7, 7)
        x_hat = self.decoder_conv(h)
        return x_hat

    def forward(self, x):
        z = self.encode(x)
        x_hat = self.decode(z)
        return x_hat

conv_model = ConvAutoencoder(latent_dim=32).to(device)
print(f"Parâmetros do ConvAutoencoder: {sum(p.numel() for p in conv_model.parameters()):,}")

## 4. Loop de Treinamento e Reconstrução

In [ ]:
def train_autoencoder(model, train_loader, epochs=10, lr=1e-3):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_history = []
    
    model.train()
    for epoch in range(1, epochs + 1):
        running_loss = 0.0
        for images, _ in train_loader:
            images = images.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, images)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
            
        epoch_loss = running_loss / len(train_loader.dataset)
        loss_history.append(epoch_loss)
        print(f"Época [{epoch}/{epochs}] - Perda MSE de Treino: {epoch_loss:.5f}")
        
    return loss_history

print("--- Treinando ConvAutoencoder (Latent Dim = 32) ---")
history_conv = train_autoencoder(conv_model, train_loader, epochs=10)

### 4.1 Visualização das Reconstruções (Original vs Reconstruído)

In [ ]:
conv_model.eval()
data_iter = iter(test_loader)
test_images, _ = next(data_iter)
test_images = test_images.to(device)

with torch.no_grad():
    reconstructions = conv_model(test_images)

# Plotar 8 amostras
plt.figure(figsize=(12, 3))
for i in range(8):
    # Imagem Original
    ax = plt.subplot(2, 8, i + 1)
    plt.imshow(test_images[i].cpu().squeeze(), cmap='gray')
    plt.title("Original")
    plt.axis('off')
    
    # Imagem Reconstruída
    ax = plt.subplot(2, 8, i + 1 + 8)
    plt.imshow(reconstructions[i].cpu().squeeze(), cmap='gray')
    plt.title("Reconstruído")
    plt.axis('off')

plt.tight_layout()
plt.show()

## 5. Inspeção do Espaço Latente (PCA & t-SNE)

Vamos extrair os vetores latentes $Z \in \mathbb{R}^{32}$ de $2.000$ imagens de teste do MNIST e projetá-los em 2D usando t-SNE.

In [ ]:
conv_model.eval()
latent_vectors = []
labels_list = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        z = conv_model.encode(images)
        latent_vectors.append(z.cpu().numpy())
        labels_list.append(labels.numpy())
        if len(np.concatenate(labels_list)) >= 2000:
            break

X_latent = np.concatenate(latent_vectors)[:2000]
y_labels = np.concatenate(labels_list)[:2000]

print(f"Formato dos vetores latentes extraídos: {X_latent.shape}")

# Projeção t-SNE
tsne = TSNE(n_components=2, random_state=42)
X_embedded = tsne.fit_transform(X_latent)

# Plotar dispersão 2D
plt.figure(figsize=(9, 7))
scatter = plt.scatter(X_embedded[:, 0], X_embedded[:, 1], c=y_labels, cmap='tab10', alpha=0.8, s=15)
plt.colorbar(scatter, label='Dígito MNIST')
plt.title('Projeção t-SNE do Espaço Latente (Bottleneck Z = 32)')
plt.xlabel('Componente t-SNE 1')
plt.ylabel('Componente t-SNE 2')
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()

## 6. Experimento de Detecção de Anomalias sem Supervisão

### 6.1 Cenário Didático
- **Treinamento Normal:** Treinamos o Autoencoder **apenas com os dígitos '1'** do MNIST (nosso *manifold* normal).
- **Teste/Avaliação:** Avaliamos o modelo em um conjunto misto contendo **dígitos '1' (normais)** e **dígitos '0', '7', '8' e '9' (anomalias)**.

In [ ]:
# Filtrar dataset para conter apenas a classe normal (dígito 1)
idx_train_1 = [i for i, (_, label) in enumerate(train_dataset) if label == 1]
normal_train_dataset = Subset(train_dataset, idx_train_1)
normal_train_loader = DataLoader(normal_train_dataset, batch_size=128, shuffle=True)

print(f"Total de amostras normais de treino (Dígito 1): {len(normal_train_dataset)}")

# Instanciar novo modelo para detecção de anomalias
anomaly_model = ConvAutoencoder(latent_dim=16).to(device)
history_anomaly = train_autoencoder(anomaly_model, normal_train_loader, epochs=10)

### 6.2 Avaliação do Erro de Reconstrução em Amostras Normais vs. Anômalas

In [ ]:
anomaly_model.eval()
criterion_none = nn.MSELoss(reduction='none')

normal_losses = []
anomalous_losses = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = anomaly_model(images)
        
        # Erro MSE por imagem
        loss_per_image = criterion_none(outputs, images).mean(dim=[1, 2, 3]).cpu().numpy()
        
        for loss_val, label in zip(loss_per_image, labels.numpy()):
            if label == 1:
                normal_losses.append(loss_val)
            else:
                anomalous_losses.append(loss_val)

normal_losses = np.array(normal_losses)
anomalous_losses = np.array(anomalous_losses)

print(f"Erro Médio Normais (Dígito 1): {normal_losses.mean():.5f} ± {normal_losses.std():.5f}")
print(f"Erro Médio Anomalias (Outros dígitos): {anomalous_losses.mean():.5f} ± {anomalous_losses.std():.5f}")

### 6.3 Plotagem da Distribuição de Erro e Análise de Threshold

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(normal_losses, bins=50, alpha=0.6, label='Normais (Dígito 1)', color='green', density=True)
plt.hist(anomalous_losses, bins=50, alpha=0.6, label='Anomalias (Outros dígitos)', color='red', density=True)

# Threshold empírico: mean(normal) + 2 * std(normal)
thresh_naive = normal_losses.mean() + 2 * normal_losses.std()
plt.axvline(thresh_naive, color='black', linestyle='--', linewidth=2, label=f'Threshold Naive (μ + 2σ = {thresh_naive:.4f})')

plt.title('Distribuição do Erro de Reconstrução (MSE)')
plt.xlabel('Erro de Reconstrução MSE')
plt.ylabel('Densidade')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### 6.4 Avaliação de Desempenho (Curva ROC-AUC e Curva Precision-Recall)

In [ ]:
# Rótulos reais: 0 = Normal, 1 = Anomalia
y_true = np.array([0] * len(normal_losses) + [1] * len(anomalous_losses))
y_scores = np.concatenate([normal_losses, anomalous_losses])

fpr, tpr, thresholds_roc = roc_curve(y_true, y_scores)
roc_auc = auc(fpr, tpr)

precision, recall, thresholds_pr = precision_recall_curve(y_true, y_scores)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Curva ROC (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('Taxa de Falsos Positivos (FPR)')
plt.ylabel('Taxa de Verdadeiros Positivos (TPR / Recall)')
plt.title('Curva ROC - Detecção de Anomalias')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(recall, precision, color='purple', lw=2, label='Curva Precision-Recall')
plt.xlabel('Recall (Sensibilidade)')
plt.ylabel('Precisão')
plt.title('Curva Precision-Recall')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 6.5 Relatório de Classificação no Threshold de Exemplo

In [ ]:
y_pred = (y_scores > thresh_naive).astype(int)
print("--- Relatório de Classificação no Threshold Naive (μ + 2σ) ---")
print(classification_report(y_true, y_pred, target_names=['Normal', 'Anômalo']))

print("Matriz de Confusão:")
print(confusion_matrix(y_true, y_pred))